In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("5-star edible sushi.csv")

columns = df.columns.tolist()

print(columns)

['project_id', 'report_month', 'month', 'year', 'sector', 'state', 'approval_year', 'project_age', 'original_duration_months', 'duration_overrun_months', 'original_cost_crore', 'revised_cost_crore', 'anticipated_cost_crore', 'current_cost', 'cumulative_expenditure_crore', 'anticipated_delay_from_original', 'remaining_org_months', 'delay_revised_months', 'delay_revisied_months', 'milestones_achieved', 'milestones_total', 'milestone_completion_percentage', 'milestone_data_reliable', 'cost_revision_percentage', 'anticipated_cost_percentage', 'exp_vs_anti_prct', 'exp_vs_org_prct', 'exp_vs_rev_prct', 'project_duration_elapsed_percentage', 'cost_growth_3m', 'cost_growth_6m', 'cost_growth_12m', 'expenditure_growth_3m', 'expenditure_growth_6m', 'expenditure_growth_12m', 'schedule_change_3m', 'schedule_change_6m', 'schedule_change_12m', 'milestone_progress_change_3m', 'milestone_progress_change_6m', 'milestone_progress_change_12m', 'expenditure_progress_gap_org', 'expenditure_progress_gap_anti'

In [2]:
df["report_month"] = pd.to_datetime(
    df["report_month"],
    errors="coerce"
)

print(df["report_month"].min())
print(df["report_month"].max())
print(df["report_month"].isna().sum())

2015-04-01 00:00:00
2024-05-01 00:00:00
0


In [3]:


def get_walk_forward_group_splits(
    df,
    n_folds=5,
    date_col="report_month",
    group_col="project_id"
):
    """
    Time-based walk-forward validation for project-month panel data.

    Each validation fold contains future months only.
    A project's earlier observations may appear in training,
    because this represents forecasting future behavior of
    an already-monitored project.
    """

    data = df.copy()

    # Ensure correct datatypes
    data[date_col] = pd.to_datetime(
        data[date_col],
        errors="coerce"
    )

    if data[date_col].isna().any():
        raise ValueError(
            f"{date_col} contains missing/invalid dates."
        )

    if data[group_col].isna().any():
        raise ValueError(
            f"{group_col} contains missing project IDs."
        )

    # Sort chronologically
    data = data.sort_values(
        [date_col, group_col]
    )

    unique_months = np.sort(
        data[date_col].dt.to_period("M").unique()
    )

    if len(unique_months) < n_folds + 1:
        raise ValueError(
            "Not enough unique months for requested number of folds."
        )

    # Split the timeline into ordered validation windows
    validation_months = np.array_split(
        unique_months[-n_folds:],
        n_folds
    )

    splits = []

    for fold, val_months in enumerate(validation_months, start=1):

        if len(val_months) == 0:
            continue

        val_start = val_months.min()

        # Everything strictly before validation period
        train_mask = (
            data[date_col].dt.to_period("M")
            < val_start
        )

        # Validation period
        val_mask = (
            data[date_col].dt.to_period("M")
            .isin(val_months)
        )

        train_idx = data.index[train_mask].to_numpy()
        val_idx = data.index[val_mask].to_numpy()

        # Safety checks
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        max_train_date = data.loc[
            train_idx, date_col
        ].max()

        min_val_date = data.loc[
            val_idx, date_col
        ].min()

        if max_train_date >= min_val_date:
            raise ValueError(
                f"Temporal leakage detected in fold {fold}."
            )

        splits.append(
            (train_idx, val_idx)
        )

        print(
            f"Fold {fold}: "
            f"TRAIN <= {max_train_date.date()} | "
            f"VALID = {min_val_date.date()} "
            f"to {data.loc[val_idx, date_col].max().date()} | "
            f"Train rows = {len(train_idx):,} | "
            f"Valid rows = {len(val_idx):,}"
        )

    if len(splits) != n_folds:
        raise ValueError(
            f"Could only create {len(splits)} folds "
            f"instead of {n_folds}."
        )

    return splits

In [4]:
splits = get_walk_forward_group_splits(
    df,
    n_folds=5
)

print("\nNumber of folds:", len(splits))

Fold 1: TRAIN <= 2022-05-01 | VALID = 2022-06-01 to 2022-06-01 | Train rows = 109,464 | Valid rows = 131
Fold 2: TRAIN <= 2022-06-01 | VALID = 2022-07-01 to 2022-07-01 | Train rows = 109,595 | Valid rows = 123
Fold 3: TRAIN <= 2022-07-01 | VALID = 2023-11-01 to 2023-11-01 | Train rows = 109,718 | Valid rows = 32
Fold 4: TRAIN <= 2023-11-01 | VALID = 2024-04-01 to 2024-04-01 | Train rows = 109,750 | Valid rows = 18
Fold 5: TRAIN <= 2024-04-01 | VALID = 2024-05-01 to 2024-05-01 | Train rows = 109,768 | Valid rows = 19

Number of folds: 5


In [5]:
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

missing_pct = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": missing_pct
})

print(missing_report)

                                     missing_count  missing_percentage
delay_revised_months                        105006           95.645204
delay_revisied_months                        94528           86.101269
exp_vs_rev_prct                              92498           84.252234
revised_cost_crore                           91874           83.683861
cost_revision_percentage                     91874           83.683861
milestone_progress_change_12m                88328           80.453970
milestone_progress_change_6m                 82656           75.287602
milestone_progress_change_3m                 79754           72.644302
schedule_progress_mismatch                   76233           69.437183
cost_rebaseline_signal                       74333           67.706559
expenditure_progress_gap_anti                74333           67.706559
expenditure_progress_gap_org                 74333           67.706559
milestone_completion_percentage              72756           66.270141
schedu

In [6]:
both = df[
    df["delay_revised_months"].notna() &
    df["delay_revisied_months"].notna()
][
    [
        "project_id",
        "report_month",
        "delay_revised_months",
        "delay_revisied_months"
    ]
]

print(both.head(50).to_string(index=False))

print("\nCorrelation:")
print(
    both[
        ["delay_revised_months", "delay_revisied_months"]
    ].corr()
)

project_id report_month  delay_revised_months  delay_revisied_months
 120100067   2015-04-01                  31.0                  -30.0
 160100231   2015-04-01                 -10.0                   12.0
 160100231   2015-05-01                 -10.0                   11.0
 160100231   2015-06-01                  36.0                  -36.0
 180100239   2015-04-01                   0.0                   23.0
 180100239   2015-05-01                   0.0                   22.0
 180100239   2015-06-01                   0.0                   21.0
 180100239   2015-07-01                   0.0                   20.0
 180100239   2015-08-01                   0.0                   10.0
 180100239   2015-09-01                   0.0                    9.0
 180100239   2015-10-01                   0.0                    8.0
 180100239   2015-11-01                   0.0                   13.0
 180100239   2015-12-01                   0.0                    6.0
 180100239   2016-01-01           

In [7]:
print(
    (both["delay_revised_months"] ==
     -both["delay_revisied_months"]).mean()
)

0.026145157916753817


In [8]:
df = df.drop(columns=["delay_revisied_months"])

print("Dropped delay_revisied_months")
print("Shape:", df.shape)

Dropped delay_revisied_months
Shape: (109787, 51)


In [9]:
print([
    c for c in df.columns
    if "delay" in c.lower()
])

['anticipated_delay_from_original', 'delay_revised_months']


In [10]:
outliers = df[
    df["cost_growth_12m"].abs() > 500
][
    [
        "project_id",
        "report_month",
        "original_cost_crore",
        "anticipated_cost_crore",
        "current_cost",
        "cost_growth_12m"
    ]
].sort_values(
    "cost_growth_12m",
    ascending=False
)

print(outliers.head(30).to_string(index=False))

project_id report_month  original_cost_crore  anticipated_cost_crore  current_cost  cost_growth_12m
 220100205   2019-11-01              1723.00                 1780.09       1780.29    177929.000000
 220100205   2019-12-01              1723.00                 3472.59       1780.29    177929.000000
 220100205   2020-02-01              1723.00                 3472.59       1780.29    177929.000000
 220100205   2020-01-01              1723.00                 3472.59       1780.29    177929.000000
 N28000106   2019-03-01               238.56                  238.56        238.56     11828.000000
 N18000164   2019-02-01               522.29                  550.18      55018.00      9900.000000
 N18000164   2019-01-01               522.29                  550.18      55018.00      9900.000000
 N18000164   2018-12-01               522.29                  550.18      55018.00      9900.000000
 N24000508   2020-09-01              1871.34                 1871.34       2754.00      3989.694090


In [11]:
negative_cost = df[
    df["cost_growth_12m"] < -50
][
    [
        "project_id",
        "report_month",
        "original_cost_crore",
        "anticipated_cost_crore",
        "current_cost",
        "cost_growth_12m"
    ]
].sort_values("cost_growth_12m")

print(negative_cost.head(30).to_string(index=False))

project_id report_month  original_cost_crore  anticipated_cost_crore  current_cost  cost_growth_12m
 220100205   2021-02-01              1723.00                 3421.35          1.00       -99.943829
 220100205   2021-01-01              1723.00                 3421.35          1.00       -99.943829
 220100205   2020-12-01              1723.00                 3472.59          1.00       -99.943829
 220100205   2019-04-01              1723.00                  645.00          1.00       -99.577006
 220100205   2019-09-01              1723.00                 1723.00          1.00       -99.577006
 220100205   2019-03-01              1723.00                  645.00          1.00       -99.577006
 220100205   2019-07-01              1723.00                 1723.00          1.00       -99.577006
 220100205   2018-12-01              1723.00                  645.00          1.00       -99.577006
 220100205   2019-02-01              1723.00                  645.00          1.00       -99.577006


In [12]:
print("current_cost <= 1:")
print((df["current_cost"] <= 1).sum())

print("\ncurrent_cost > 10x original:")
print(
    (
        df["current_cost"] >
        df["original_cost_crore"] * 10
    ).sum()
)

print("\ncurrent_cost statistics:")
print(df["current_cost"].describe())

current_cost <= 1:
35

current_cost > 10x original:
72

current_cost statistics:
count    109787.000000
mean       1355.339170
std        4073.031278
min           1.000000
25%         270.000000
50%         480.210000
75%        1030.650000
max      108000.000000
Name: current_cost, dtype: float64


In [13]:
suspicious_current = df[
    (df["current_cost"] <= 1) |
    (df["current_cost"] > df["original_cost_crore"] * 10)
][
    [
        "project_id",
        "report_month",
        "original_cost_crore",
        "anticipated_cost_crore",
        "current_cost",
        "cost_growth_3m",
        "cost_growth_6m",
        "cost_growth_12m"
    ]
]

print(suspicious_current.head(50).to_string(index=False))
print("\nSuspicious rows:", len(suspicious_current))

project_id report_month  original_cost_crore  anticipated_cost_crore  current_cost  cost_growth_3m  cost_growth_6m  cost_growth_12m
 220100133   2018-07-01               2500.0                27949.00       27949.0     1017.960000       42.852032      1017.960000
 220100133   2018-09-01               2500.0                27949.00       27949.0             NaN     1017.960000      1017.960000
 220100133   2018-10-01               2500.0                27949.00       27949.0        0.000000     1017.960000        42.852032
 220100133   2018-11-01               2500.0                27949.00       27949.0             NaN             NaN        42.852032
 220100133   2019-06-01               2500.0                27949.00       27949.0     1017.960000     1017.960000              NaN
 220100133   2019-07-01               2500.0                27949.00       27949.0     1017.960000     1017.960000         0.000000
 220100133   2019-08-01               2500.0                27949.00       2

In [14]:
suspicious = df[
    (df["current_cost"] <= 1) |
    (df["current_cost"] > 10 * df["original_cost_crore"])
].copy()

cols = [
    "project_id",
    "report_month",
    "original_cost_crore",
    "revised_cost_crore",
    "anticipated_cost_crore",
    "current_cost",
    "cost_growth_3m",
    "cost_growth_6m",
    "cost_growth_12m"
]

print(suspicious[cols].sort_values("current_cost").to_string(index=False))

project_id report_month  original_cost_crore  revised_cost_crore  anticipated_cost_crore  current_cost  cost_growth_3m  cost_growth_6m  cost_growth_12m
 220100205   2018-11-01              1723.00                1.00                  645.00          1.00             NaN             NaN       -99.577006
 220100205   2018-12-01              1723.00                1.00                  645.00          1.00      -99.577006             NaN       -99.577006
 220100205   2019-01-01              1723.00                1.00                  645.00          1.00      -99.577006      -99.577006       -99.577006
 220100205   2022-02-01              1723.00                1.00                 3421.35          1.00        0.000000        0.000000         0.000000
 220100205   2022-01-01              1723.00                1.00                 3421.35          1.00        0.000000        0.000000         0.000000
 220100205   2021-12-01              1723.00                1.00                 3421.35

In [15]:
print(
    df.loc[df["current_cost"] <= 1, cols]
      .to_string(index=False)
)

project_id report_month  original_cost_crore  revised_cost_crore  anticipated_cost_crore  current_cost  cost_growth_3m  cost_growth_6m  cost_growth_12m
 220100205   2018-11-01               1723.0                 1.0                  645.00           1.0             NaN             NaN       -99.577006
 220100205   2018-12-01               1723.0                 1.0                  645.00           1.0      -99.577006             NaN       -99.577006
 220100205   2019-01-01               1723.0                 1.0                  645.00           1.0      -99.577006      -99.577006       -99.577006
 220100205   2019-02-01               1723.0                 1.0                  645.00           1.0        0.000000             NaN       -99.577006
 220100205   2019-03-01               1723.0                 1.0                  645.00           1.0        0.000000      -99.577006       -99.577006
 220100205   2019-04-01               1723.0                 1.0                  645.00

In [16]:
print(
    df.loc[df["current_cost"] > 10 * df["original_cost_crore"], cols]
      .sort_values("current_cost", ascending=False)
      .to_string(index=False)
)

project_id report_month  original_cost_crore  revised_cost_crore  anticipated_cost_crore  current_cost  cost_growth_3m  cost_growth_6m  cost_growth_12m
 N18000164   2019-02-01               522.29            55018.00                  550.18      55018.00    10433.994524             NaN      9900.000000
 N18000164   2018-12-01               522.29            55018.00                  550.18      55018.00     9900.000000             NaN      9900.000000
 N18000164   2019-01-01               522.29            55018.00                  550.18      55018.00     9900.000000     9900.000000      9900.000000
 220100133   2018-07-01              2500.00            27949.00                27949.00      27949.00     1017.960000       42.852032      1017.960000
 220100133   2019-06-01              2500.00            27949.00                27949.00      27949.00     1017.960000     1017.960000              NaN
 220100133   2018-09-01              2500.00            27949.00                27949.00

In [17]:
print(df[
    [
        "original_cost_crore",
        "revised_cost_crore",
        "anticipated_cost_crore",
        "current_cost"
    ]
].head(20).to_string(index=False))

 original_cost_crore  revised_cost_crore  anticipated_cost_crore  current_cost
             8692.00            12291.00                12291.00      12291.00
             8692.00            12291.00                12291.00      12291.00
             8692.00            12291.00                12291.00      12291.00
             8692.00            12291.00                12291.00      12291.00
             8692.00            12291.00                12291.00      12291.00
             8692.00            12291.00                12291.00      12291.00
              429.82             3955.21                 3955.21       3955.21
              429.82             3955.21                 3955.21       3955.21
              429.82             3955.21                 3955.21       3955.21
              429.82             3955.21                 3955.21       3955.21
              578.62                 NaN                  578.62        578.62
              578.62                 NaN            

In [18]:
# Make a clean revised-cost copy
df["revised_cost_clean"] = df["revised_cost_crore"]

# Flag suspicious revised costs
bad_revised = (
    df["revised_cost_clean"].notna() &
    (df["revised_cost_clean"] <= 1) &
    (df["original_cost_crore"] > 1)
)

print("Suspicious revised costs:", bad_revised.sum())

# Treat those suspicious values as missing
df.loc[bad_revised, "revised_cost_clean"] = np.nan

# Current cost = revised cost if valid, otherwise original cost
df["current_cost"] = (
    df["revised_cost_clean"]
    .fillna(df["original_cost_crore"])
)

Suspicious revised costs: 35


In [19]:
df = df.sort_values(["project_id", "report_month"])

for months in [3, 6, 12]:
    old_cost = df.groupby("project_id")["current_cost"].shift(months)

    df[f"cost_growth_{months}m"] = (
        (df["current_cost"] - old_cost) / old_cost
    ) * 100

In [20]:
print(df[[
    "original_cost_crore",
    "revised_cost_crore",
    "revised_cost_clean",
    "current_cost"
]].head(20))

print("\nCurrent cost <= 1:")
print((df["current_cost"] <= 1).sum())

print("\nCost growth statistics:")
print(df["cost_growth_12m"].describe())

    original_cost_crore  revised_cost_crore  revised_cost_clean  current_cost
0               8692.00            12291.00            12291.00      12291.00
1               8692.00            12291.00            12291.00      12291.00
2               8692.00            12291.00            12291.00      12291.00
3               8692.00            12291.00            12291.00      12291.00
4               8692.00            12291.00            12291.00      12291.00
5               8692.00            12291.00            12291.00      12291.00
6                429.82             3955.21             3955.21       3955.21
7                429.82             3955.21             3955.21       3955.21
8                429.82             3955.21             3955.21       3955.21
9                429.82             3955.21             3955.21       3955.21
10               578.62                 NaN                 NaN        578.62
11               578.62                 NaN                 NaN 

In [21]:
print("Current cost <= 1:", (df["current_cost"] <= 1).sum())

print("\nCurrent cost > 10x original:",
      (df["current_cost"] > 10 * df["original_cost_crore"]).sum())

print("\nCost growth 12m:")
print(df["cost_growth_12m"].describe())

Current cost <= 1: 0

Current cost > 10x original: 72

Cost growth 12m:
count    78198.000000
mean         8.096561
std        107.538169
min        -99.006943
25%          0.000000
50%          0.000000
75%          0.000000
max      11828.000000
Name: cost_growth_12m, dtype: float64


In [22]:
df = df.sort_values(["project_id", "report_month"])

for months in [3, 6, 12]:
    old_cost = df.groupby("project_id")["current_cost"].shift(months)

    df[f"cost_growth_{months}m"] = (
        (df["current_cost"] - old_cost) / old_cost
    ) * 100

In [23]:
df = df.sort_values(["project_id", "report_month"])

for months in [3, 6, 12]:
    old_cost = df.groupby("project_id")["current_cost"].shift(months)

    df[f"cost_growth_{months}m"] = (
        (df["current_cost"] - old_cost) / old_cost
    ) * 100

In [24]:
print(df[[
    "cost_growth_3m",
    "cost_growth_6m",
    "cost_growth_12m"
]].describe())

       cost_growth_3m  cost_growth_6m  cost_growth_12m
count   101384.000000    93383.000000     78198.000000
mean         2.726930        4.671350         8.096561
std         79.205363       90.278676       107.538169
min        -99.006943      -99.050693       -99.006943
25%          0.000000        0.000000         0.000000
50%          0.000000        0.000000         0.000000
75%          0.000000        0.000000         0.000000
max      11828.000000    11828.000000     11828.000000


In [25]:
print(
    df.loc[
        df["cost_growth_12m"] > 500,
        [
            "project_id",
            "report_month",
            "original_cost_crore",
            "revised_cost_crore",
            "anticipated_cost_crore",
            "current_cost",
            "cost_growth_12m"
        ]
    ]
    .sort_values("cost_growth_12m", ascending=False)
    .head(20)
    .to_string(index=False)
)

project_id report_month  original_cost_crore  revised_cost_crore  anticipated_cost_crore  current_cost  cost_growth_12m
 N28000106   2019-04-01               238.56                 NaN                  238.56        238.56     11828.000000
 N18000164   2018-12-01               522.29            55018.00                  550.18      55018.00      9900.000000
 N18000164   2019-02-01               522.29            55018.00                  550.18      55018.00      9900.000000
 N18000164   2019-01-01               522.29            55018.00                  550.18      55018.00      9900.000000
 N24000508   2020-08-01              1871.34             2754.00                 1871.34       2754.00      3989.694090
 N24000508   2020-12-01              1871.34             2754.00                 1871.34       2754.00      3989.694090
 N24000508   2020-09-01              1871.34             2754.00                 1871.34       2754.00      3989.694090
 N24000508   2021-01-01              187

In [26]:
# Flag extreme cost-growth observations for later analysis.
# DO NOT delete them.
df["extreme_cost_growth_flag"] = (
    df["cost_growth_12m"].abs() > 500
).astype(int)

print("Extreme cost-growth rows:",
      df["extreme_cost_growth_flag"].sum())

Extreme cost-growth rows: 258


In [27]:
df.drop(columns=["revised_cost_clean"], errors="ignore", inplace=True)

In [28]:
import numpy as np

numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols] = df[numeric_cols].replace(
    [np.inf, -np.inf], np.nan
)

print("Infinite values:",
      np.isinf(df[numeric_cols]).sum().sum())

Infinite values: 0


In [29]:
df.to_parquet(
    "paimana_train_v1.parquet",
    index=False
)

print("Saved:", df.shape)

Saved: (109787, 52)
